In [0]:
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Row
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

start_date = '2021-01-01'
end_date = '2026-06-17'

In [0]:
# Brunch À La Carte — Item_Description as they appear in cpx_d_item
brunch_items = [
    'PEACH DON\'T KILL MY VIBE',   # HO CODE: 38300  | Menu: $9
    'CLASSIC MIMOSA',               # HO CODE: 39693  | Menu: $10
    'GRAPEFRUIT MIMOSA',            # HO CODE: 39585  | Menu: $10
    'PICKLED CAESAR',               # HO CODE: 38341  | Menu: $12.50
    'BRUNCH MIMOSA',                # HO CODE: 22047  | Menu: $49 (Mimosa Bottle)
    'BRUNCH POWER UP',              # HO CODE: 22309  | Menu: $10 (add-on)
    'BOC POUTINE',                  # HO CODE: 29103  | Menu: $14
    'CLASSIC BREAKFAST',            # HO CODE: 29120  | Menu: $13
    'KNIFE FORK SANDWICH',          # HO CODE: 29118  | Menu: $15
    'TRAD EGGS BENEDICT',           # HO CODE: 29114  | Menu: $15
    'GOAT CHEESE BENEDICT',         # HO CODE: 29119  | Menu: $16
    'KIDS GAMER BREAKFAST',         # HO CODE: 29101  | Menu: $8
]

# Categorized by menu section
brunch_bins = {
    'Cocktails': [
        'PEACH DON\'T KILL MY VIBE',
        'CLASSIC MIMOSA',
        'GRAPEFRUIT MIMOSA',
        'PICKLED CAESAR',
        'BRUNCH MIMOSA',
    ],
    'Sleep In Specials': [
        'BRUNCH POWER UP',
        'BOC POUTINE',
        'CLASSIC BREAKFAST',
        'KNIFE FORK SANDWICH',
        'TRAD EGGS BENEDICT',
        'GOAT CHEESE BENEDICT',
    ],
    'Kids': [
        'KIDS GAMER BREAKFAST',
    ],
}

In [0]:
# Create category mapping DataFrame
mapping_data = [
    Row(Item_Description=item, category=cat) 
    for cat, items in brunch_bins.items() 
    for item in items
]

bin_mapping_df = spark.createDataFrame(mapping_data)

T_brunch = (
    T_trans
    .join(
        T_date.select("dateid", "datefull"),
        T_date.dateid == T_trans.VisitDateId,
        "left"
    )
    .join(
        T_location.select(
            "LocationID",
            "Province_CD",
            "LC_Trade_Brand_Description",
            "LC_Location_Type_Description",
            "Location_Name"
        ),
        T_location.LocationID == T_trans.LocationId,
        "left"
    )
    .join(
        T_item.select(
            "itemskey",
            "Item_Description",
            "Item_Class_Description1"
        ),
        T_item.itemskey == T_trans.ItemSkey,
        "left"
    )
    .join(bin_mapping_df, on="Item_Description", how="left")

    .filter(F.col("datefull").between(start_date, end_date))
    .filter(F.col("LC_Location_Type_Description").like("%Restaurant%"))
    # .filter(F.col("Location_Name").like("%Granville%"))
    .filter(F.col("Item_Description").isin(brunch_items))

    .withColumn("Year", F.year("datefull").cast("string"))
)

# # Summary by Item and Year
# T_brunch_by_year = (
#     T_brunch
#     .groupBy("Year", "Item_Description", "category")
#     .agg(
#         F.sum("Quantity").alias("Total_Quantity"),
#         F.sum("Net_Revenue").alias("Total_Revenue"),
#         F.countDistinct("CDE_ID").alias("Unique_Customers"),
#     )
#     .orderBy("Year", "category", "Item_Description")
# )

# T_brunch_by_year.display()

In [0]:
# Check if brunch_items match Item_Description in the item table
# Filter transactions to 2025 at Granville TRR
T_check = (
    T_trans
    .join(T_date.select("dateid", "datefull"), T_date.dateid == T_trans.VisitDateId, "left")
    .join(T_item.select("itemskey", "Item_Description"), T_item.itemskey == T_trans.ItemSkey, "left")
    .join(T_location.select("LocationID", "Location_Name", "LC_Location_Type_Description"), T_location.LocationID == T_trans.LocationId, "left")
    .filter(F.year("datefull") == 2025)
    .filter(F.col("LC_Location_Type_Description").like("%Restaurant%"))
    .filter(F.col("Item_Description").isin(brunch_items))
    .groupBy("Item_Description")
    .agg(
        F.count("*").alias("Txn_Count"),
        F.sum("Quantity").alias("Total_Qty"),
        F.sum("Net_Revenue").alias("Total_Revenue"),
        F.min("datefull").alias("First_Seen"),
        F.max("datefull").alias("Last_Seen"),
    )
    .orderBy("Item_Description")
)

print(f"Matched {T_check.count()} of {len(brunch_items)} brunch items in 2025:")
T_check.display()

In [0]:
T_check.explain()

In [0]:
from pyspark.sql import functions as F

t = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
d = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
i = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
l = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

brunch_items = (
    i.Item_Description.like("%BRUNCH%POWER%")
    | i.Item_Description.like("%BOC%POUTINE%")
    | i.Item_Description.like("%CLASSIC%BREAKFAST%")
    | i.Item_Description.like("%KNIFE%FORK%")
    | i.Item_Description.like("%EGGS%BENEDICT%")
    | i.Item_Description.like("%GOAT%CHEESE%BENEDICT%")
    | i.Item_Description.like("%GAMER%BREAKFAST%")
)

result = (
    t
    .join(d, d.dateid == t.VisitDateId, "inner")
    .join(i, i.itemskey == t.ItemSkey, "inner")
    .join(l, l.LocationID == t.LocationId, "inner")
    .where(
        (F.year(d.datefull) == 2021)
        & l.LC_Location_Type_Description.like("%Restaurant%")
        & brunch_items
    )
    .groupBy(i.Item_Description, l.Location_Name)
    .agg(
        F.count("*").alias("Txn_Count"),
        F.sum(t.Quantity).alias("Total_Qty"),
        F.sum(t.Net_Revenue).alias("Total_Revenue"),
        F.min(d.datefull).alias("First_Seen"),
        F.max(d.datefull).alias("Last_Seen"),
    )
    .orderBy(i.Item_Description, l.Location_Name)
)

In [0]:
result.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

spark.conf.set("spark.sql.shuffle.partitions", 32)

t = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
d = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
i = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
l = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

brunch_items = (
    i.Item_Description.like("%BRUNCH%POWER%")
    | i.Item_Description.like("%BOC%POUTINE%")
    | i.Item_Description.like("%CLASSIC%BREAKFAST%")
    | i.Item_Description.like("%KNIFE%FORK%")
    | i.Item_Description.like("%EGGS%BENEDICT%")
    | i.Item_Description.like("%GOAT%CHEESE%BENEDICT%")
    | i.Item_Description.like("%GAMER%BREAKFAST%")
)

result = (
    t
    .join(broadcast(d), d.dateid == t.VisitDateId, "inner")
    .join(broadcast(i), i.itemskey == t.ItemSkey, "inner")
    .join(broadcast(l), l.LocationID == t.LocationId, "inner")
    .where(
        (F.year(d.datefull) == 2021)
        & l.LC_Location_Type_Description.like("%Restaurant%")
        & brunch_items
    )
    .groupBy(i.Item_Description, l.Location_Name)
    .agg(
        F.count("*").alias("Txn_Count"),
        F.sum(t.Quantity).alias("Total_Qty"),
        F.sum(t.Net_Revenue).alias("Total_Revenue"),
        F.min(d.datefull).alias("First_Seen"),
        F.max(d.datefull).alias("Last_Seen"),
    )
    .orderBy(i.Item_Description, l.Location_Name)
)


In [0]:
result.display()

In [0]:
%sql
-- Optimized:
--   1. Broadcast hints moved to SELECT clause (/*+ BROADCAST(...) */ is the correct Spark SQL syntax;
--      inline /*broadcast*/ comments after JOIN are ignored)
--   2. YEAR() replaced with date range — avoids wrapping the column in a function so
--      zone maps / stats-based skipping can prune files before scanning
--   3. Dimension filters pushed into ON clauses — Spark prunes before joining the fact table
SELECT /*+ BROADCAST(d), BROADCAST(i), BROADCAST(l) */
  i.Item_Description,
  l.Location_Name,
  COUNT(*)              AS Txn_Count,
  SUM(t.Quantity)       AS Total_Qty,
  SUM(t.Net_Revenue)    AS Total_Revenue,
  MIN(d.datefull)       AS First_Seen,
  MAX(d.datefull)       AS Last_Seen
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
  ON  d.dateid  = t.VisitDateId
  AND d.datefull BETWEEN '2021-01-01' AND '2021-12-31'   -- date range enables zone-map skipping
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
  ON  i.itemskey = t.ItemSkey
  AND i.Item_Description IN (
        'BRUNCH POWER UP',
        'BOC POUTINE',
        'CLASSIC BREAKFAST',
        'KNIFE FORK SANDWICH',
        'TRAD EGGS BENEDICT',
        'GOAT CHEESE BENEDICT',
        'KIDS GAMER BREAKFAST'
      )
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
  ON  l.LocationID = t.LocationId
  AND l.LC_Location_Type_Description LIKE '%Restaurant%'
GROUP BY i.Item_Description, l.Location_Name
ORDER BY i.Item_Description, l.Location_Name

In [0]:
%sql
-- EXPLAIN shows the physical plan Spark chose.
-- Key things to verify:
--   1. BroadcastHashJoin (not SortMergeJoin) for all three dimension joins
--   2. Filters appear early in the Scan stage (predicate pushdown confirmed)
--   3. PartialAggregate pushed before the join (partial_sum / partial_count present)
EXPLAIN
SELECT /*+ BROADCAST(d), BROADCAST(i), BROADCAST(l) */
  i.Item_Description,
  l.Location_Name,
  COUNT(*)              AS Txn_Count,
  SUM(t.Quantity)       AS Total_Qty,
  SUM(t.Net_Revenue)    AS Total_Revenue,
  MIN(d.datefull)       AS First_Seen,
  MAX(d.datefull)       AS Last_Seen
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
  ON  d.dateid  = t.VisitDateId
  AND d.datefull BETWEEN '2021-01-01' AND '2021-12-31'
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
  ON  i.itemskey = t.ItemSkey
  AND i.Item_Description IN (
        'BRUNCH POWER UP',
        'BOC POUTINE',
        'CLASSIC BREAKFAST',
        'KNIFE FORK SANDWICH',
        'TRAD EGGS BENEDICT',
        'GOAT CHEESE BENEDICT',
        'KIDS GAMER BREAKFAST'
      )
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
  ON  l.LocationID = t.LocationId
  AND l.LC_Location_Type_Description LIKE '%Restaurant%'
GROUP BY i.Item_Description, l.Location_Name
ORDER BY i.Item_Description, l.Location_Name

In [0]:
%sql
-- EXPLAIN shows the physical plan Spark chose.
-- Key things to verify:
--   1. BroadcastHashJoin (not SortMergeJoin) for all three dimension joins
--   2. PartialAggregate pushed before the join ("partial_sum", "partial_count")
--   3. Filter predicates shown early in the scan stage (predicate pushdown working)
EXPLAIN
SELECT /*+ BROADCAST(d), BROADCAST(i), BROADCAST(l) */
  i.Item_Description,
  l.Location_Name,
  COUNT(*)              AS Txn_Count,
  SUM(t.Quantity)       AS Total_Qty,
  SUM(t.Net_Revenue)    AS Total_Revenue,
  MIN(d.datefull)       AS First_Seen,
  MAX(d.datefull)       AS Last_Seen
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
  ON  d.dateid  = t.VisitDateId
  AND d.datefull BETWEEN '2021-01-01' AND '2021-12-31'
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_item i
  ON  i.itemskey = t.ItemSkey
  AND i.Item_Description IN (
        'BRUNCH POWER UP',
        'BOC POUTINE',
        'CLASSIC BREAKFAST',
        'KNIFE FORK SANDWICH',
        'TRAD EGGS BENEDICT',
        'GOAT CHEESE BENEDICT',
        'KIDS GAMER BREAKFAST'
      )
INNER JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
  ON  l.LocationID = t.LocationId
  AND l.LC_Location_Type_Description LIKE '%Restaurant%'
GROUP BY i.Item_Description, l.Location_Name
ORDER BY i.Item_Description, l.Location_Name